# E1 — feature library spot-check (PLAN.md step 4)

Visual check that the per-cell features in `src/features.py` are aligned with the cost map
and mean what their names say, on three test instances.

Nothing is computed here that a result depends on: the arrays come from
`results/features_{split}.npz` (built by `scripts/build_features.py`, i.e. `make features`)
and the figure is drawn by `scripts/plot_feature_spotcheck.py` (`make spotcheck`).
This notebook only displays them — see the working rules, "notebooks are exploration only".

Run from the repository root with the project venv:
`.venv/bin/python -m jupyterlab notebooks/01_features_spotcheck.ipynb`

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np

from scripts.plot_feature_spotcheck import correlations, load_archive, variance_split
from src.features import FEATURE_NAMES, TOPOLOGY_LABELS, WALL_ZONE_LABELS

archive = load_archive("test")
print(f"{len(archive['instance_ids'])} instances, {len(FEATURE_NAMES)} features")
print(FEATURE_NAMES)

## The figure

Regenerate it in-process (the same code path `make spotcheck` uses) and display it.

In [ ]:
%matplotlib inline
from pathlib import Path

from IPython.display import Image, display

from scripts.plot_feature_spotcheck import render

rows = [int(np.flatnonzero(archive["instance_ids"] == i)[0]) for i in (0, 1, 2)]
path = render(archive, rows, Path("../results/e1_features_spotcheck.png"))
display(Image(filename=str(path)))

**What to look for.**

* `cost map` carries a visible low-cost band between start and goal, and high-frequency
  texture elsewhere — including over obstacle cells, which the search masks away.
* `geodesic to goal` is white (NaN) over cells with no path to the goal; these maps have
  many (goals are sampled from the largest component only).
* `chebyshev to goal` is a smooth radial gradient — the upstream heuristic, obstacle-blind.
* `on ANY optimal path` is a thick band; `reference path` is one staircase inside it.
  That gap is the tie-heaviness of these maps, and the reason step 5 fits both.

## Correlation diagnostic

Pooled and within-instance correlation of each feature with the cost map.
This is alignment evidence, **not** the surrogate fit — that is step 5.

In [ ]:
spread = variance_split(archive)
print(f"cost map: sd {spread['overall_sd']:.4f} "
      f"(between-instance {spread['between_sd']:.4f}, within {spread['within_sd']:.4f})")
print()
print(f"| {'feature':28} | {'pooled r':>8} | {'within r':>8} |")
for row in correlations(archive):
    print(f"| {row['name']:28} | {row['pooled_r']:8.3f} | {row['within_r']:8.3f} |")

In [ ]:
# class balance of the two partitions, over free reachable cells
mask = (archive["is_free"] > 0) & (archive["reachable"] > 0)
for name, labels in (("topology_class", TOPOLOGY_LABELS), ("wall_zone", WALL_ZONE_LABELS)):
    values = archive[name][mask]
    print(name)
    for code_, label in labels.items():
        share = float((values == code_).mean())
        if share:
            print(f"   {label:15} {share:6.1%}")
    print()